# 03 · Fraud Rings and Pathfinding

The second showpiece: the questions SQL effectively *cannot* answer.

A **fraud ring** is a cycle — account A pays B pays C pays back to A. In relational
algebra that is a nightmare; in Cypher it is a short pattern:
`MATCH p = (a)-[:PAYS*3..5]->(a)`. We find every planted ring, then do **shortest path**
three ways. All of it runs inside the same Postgres that holds the transactions — no
Neo4j, no copy, no ETL.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import networkx as nx

from src.age.connect import age_available, get_connection
from src.baselines.networkx_graph import NetworkxGraph
from src.config import load_settings
from src.datasets.graphs import KNOWS, PAYS, GraphData, all_graphs
from src.evaluation.correctness import all_rings_found, rings_to_canonical
from src.outputs import figure_path, save_json
from src.queries.paths import shortest_path_age, shortest_path_cypher
from src.queries.paths import shortest_path_networkx, shortest_path_sql
from src.queries.rings import find_rings_age, find_rings_networkx, find_rings_sql, ring_cypher
from src.visualisation import draw_graph

settings = load_settings()
AGE = age_available(settings)
social = all_graphs(settings)["social_txn"]
conn = get_connection(settings) if AGE else None
START = 0
print("AGE:", AGE, "| planted rings:", len(social.rings))

AGE: True | planted rings: 6


## Find every fraud ring

The generator plants rings so they are the **only** cycles in `PAYS`, so all three
approaches should return exactly the planted set — and every planted ring must be found.

AGE and NetworkX run on the full transaction graph. The recursive-SQL cycle search,
though, enumerates paths from every node and is ruinous on a large graph — so we first
scope it to the **strongly-connected core** (a cheap `O(V+E)` step; the acyclic bulk has
no cycles anyway). That the SQL approach needs this extra scaffolding at all is part of
the point.

In [2]:
MIN_LEN, MAX_LEN = 3, 5
planted = rings_to_canonical(social.rings)

# Scope SQL to the strongly-connected core so the recursive CTE stays tractable.
pays = social.to_networkx(PAYS)
core_ids = {n for comp in nx.strongly_connected_components(pays) if len(comp) > 1 for n in comp}
core = GraphData(
    "pays_core",
    [n for n in social.nodes if n.node_id in core_ids],
    [e for e in social.edges if e.label == PAYS and e.src in core_ids and e.dst in core_ids],
    rings=social.rings,
)
print(f"SCC core: {len(core.nodes)} of {len(social.nodes)} accounts carry all cycles")

sql_rings = find_rings_sql(core, MIN_LEN, MAX_LEN, PAYS)
nx_rings = find_rings_networkx(social, MIN_LEN, MAX_LEN, PAYS)
age_rings = (
    find_rings_age(conn, settings.graphs.social, MIN_LEN, MAX_LEN, PAYS)
    if AGE else set(nx_rings)
)

print(f"planted: {len(planted)} | AGE: {len(age_rings)} | "
      f"SQL: {len(sql_rings)} | NetworkX: {len(nx_rings)}")
assert all_rings_found(age_rings, social.rings), "AGE missed a planted ring"
assert all_rings_found(sql_rings, social.rings), "SQL missed a planted ring"
assert all_rings_found(nx_rings, social.rings), "NetworkX missed a planted ring"
assert planted == sql_rings == nx_rings == age_rings, "planted rings are the only cycles"

print("\nThe Cypher cycle pattern:\n ", ring_cypher(MIN_LEN, MAX_LEN, PAYS))
rings_out = {
    "pattern": ring_cypher(MIN_LEN, MAX_LEN, PAYS),
    "planted_count": len(planted),
    "found": {"AGE": len(age_rings), "SQL": len(sql_rings), "NetworkX": len(nx_rings)},
    "all_planted_found": True,
    "rings": sorted(list(r) for r in planted),
}
print("saved ->", save_json("03_rings.json", rings_out))

SCC core: 24 of 400 accounts carry all cycles
planted: 6 | AGE: 6 | SQL: 6 | NetworkX: 6

The Cypher cycle pattern:
  MATCH p = (a:Account)-[:PAYS*3..5]->(a) RETURN nodes(p) AS ring
saved -> /workspaces/apache_age_101/outputs/data/03_rings.json


## Shortest path between two accounts

Cypher, the recursive SQL CTE, and NetworkX should all agree on the path length. Both the
SQL CTE and AGE's variable-length match are **bounded** to the true distance — neither has
an early-stopping shortest-path operator, so an unbounded search enumerates the universe
of paths and drowns on a dense graph.

In [3]:
G = NetworkxGraph(social, KNOWS).graph
lengths = nx.single_source_shortest_path_length(G, START, cutoff=4)
lengths.pop(START, None)
target = max(lengths, key=lengths.get)
cap = lengths[target]  # bound both the SQL CTE and AGE's *1..cap to the true distance

nx_path = shortest_path_networkx(social, START, target, KNOWS)
sql_path = shortest_path_sql(social, START, target, KNOWS, max_hops=cap)
age_path = (
    shortest_path_age(conn, settings.graphs.social, START, target, KNOWS, max_hops=cap)
    if AGE else None
)

print(f"shortest path {START} -> {target} ({cap} hops):")
print("  NetworkX:", nx_path)
print("  SQL     :", sql_path)
print("  AGE     :", age_path)
assert sql_path is not None and nx_path is not None
assert len(sql_path) == len(nx_path)
if AGE:
    assert age_path is not None and len(age_path) == len(nx_path)

save_json("03_shortest_path.json", {
    "cypher": shortest_path_cypher(START, target, KNOWS, max_hops=cap),
    "source": START,
    "target": target,
    "length_hops": len(nx_path) - 1,
    "paths": {"AGE": age_path, "SQL": sql_path, "NetworkX": nx_path},
    "lengths_agree": True,
})

shortest path 0 -> 20 (4 hops):
  NetworkX: [0, 180, 206, 326, 20]
  SQL     : [0, 192, 8, 38, 20]
  AGE     : [0, 180, 206, 326, 20]


PosixPath('/workspaces/apache_age_101/outputs/data/03_shortest_path.json')

## Figure: the planted rings

In [4]:
ring_nodes = {n for r in social.rings for n in r}
ring_sub = social.to_networkx(PAYS).subgraph(ring_nodes)
p = draw_graph(
    ring_sub,
    figure_path("03_fraud_rings.png"),
    title="Planted payment rings (cycles in PAYS)",
    highlight=ring_nodes,
)
print(f"{ring_sub.number_of_nodes()} ring accounts, {ring_sub.number_of_edges()} edges -> {p}")

24 ring accounts, 24 edges -> /workspaces/apache_age_101/outputs/figures/03_fraud_rings.png


## Figure: the shortest path

In [5]:
path_sub = social.to_networkx(KNOWS).subgraph(nx_path)
p = draw_graph(
    path_sub,
    figure_path("03_shortest_path.png"),
    title=f"Shortest KNOWS path {START} -> {target} ({len(nx_path) - 1} hops)",
    highlight={START, target},
)
print("saved ->", p)
if conn is not None:
    conn.close()
print("Done. Next: 04 — the arena.")

saved -> /workspaces/apache_age_101/outputs/figures/03_shortest_path.png
Done. Next: 04 — the arena.


**Takeaway.** Cycle detection — the fraud-ring signature — is a one-line Cypher
pattern and a planted-truth match across all three engines, run right next to the
transactions. Shortest path agrees too, but notice both SQL *and* AGE must be *bounded* or
they drown in paths. Next we score all of this honestly in the arena.